# W13A · 캠퍼스 팝업 콘텐츠 제작 스튜디오

행사 운영팀의 주니어 개발자입니다. 운영 메모와 FAQ로 공지 초안을 만드는 앱을 맡았습니다. **데이터 → 검색 → 실제 학습 → 통제 비교 → 앱 적용**을 하나씩 연결합니다. 모델이 틀린 곳을 발견한 기록도 과제의 성과입니다.

| 수업 75분 | 직접 할 일 | 과제 1에 남는 증거 |
|---|---|---|
| 00–08분 · 00절 | 독립영화 앱의 초안과 운영 원문 대조 | 원문과 화면의 역할 |
| 08–20분 · 01절 | 사진 워크숍의 응답 라벨·마스크 구성 | 데이터 분할·라벨 해시 |
| 20–32분 · 02절 | 북토크의 E5 점수 계산과 문서 결정 | 사용 문서·프롬프트 |
| 32–49분 · 03절 | Qwen2 LoRA 4회 업데이트와 재로딩 | 가중치 동결·학습 추적 |
| 49–66분 · 04–05절 | A/B/C 통제 비교·개인 D 가설 | 검증표·기본값 |
| 66–75분 · 06–07절 | 참가비 갱신·앱 반영·ZIP | 앱 상태·제출 기록 |

Colab에서 **파일 → Drive에 사본 저장** 후 하드웨어 가속기를 없음(CPU)으로 둡니다. 아래 왼쪽 ▶를 누릅니다. **W13A 준비 완료 · 공개 자료 32건**이 뜨면 시작합니다. 수업과 과제에서 이 사본을 계속 사용하세요.


In [ ]:
# @title 00 준비 · 왼쪽 ▶ { display-mode: "form" }
from base64 import b64decode
from hashlib import sha256
from html import escape
from inspect import getsource
from importlib.metadata import PackageNotFoundError, version
from importlib.util import find_spec
from io import BytesIO
from pathlib import Path
from tempfile import mkdtemp
from zipfile import ZipFile
import json
import subprocess
import sys

# 데이터·앱·가벼운 어댑터와 실측 기록을 이 사본의 임시 폴더에 풉니다.
bundle = (
    '...'
)
binary = b64decode(bundle)
assert sha256(binary).hexdigest() == "c9003cb7cdfcf10317d80d0080b663fd7ac85199778aeffc8bbada3cb2e1743f"
directory = Path(mkdtemp(prefix="w13a-design-"))
with ZipFile(BytesIO(binary)) as archive:
    for name in archive.namelist():
        destination = directory / name
        assert destination.resolve().is_relative_to(directory.resolve())
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(name))
sys.path.insert(0, str(directory))
if find_spec("ipywidgets") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "ipywidgets==8.1.8"])
try:
    from google.colab import output
except ImportError:
    pass
else:
    output.enable_custom_widget_manager()
from IPython.display import HTML, Markdown, display
from luna_genai.design_app import CourseApp, LABELS, STATES, preview_view, show_widget, table_html
from luna_genai.event_design import (
    ASSETS, CORE, FIELDS, UPDATE_EVENT, DesignStudio, configs, expected_facts, field_rows, format_sources,
    mask_witness, preview, prompt, summarize, target, training_spec, training_documents,
)

studio = DesignStudio()
bank = studio.events
ref = studio.reference
train = [event for event in bank.values() if event["split"] == "train"]

def prepare_live_packages():
    from packaging.version import Version

    def installed(name):
        try:
            return version(name)
        except PackageNotFoundError:
            return None

    desired = {"transformers": "5.15.1", "peft": "0.21.2"}
    prior = installed("torchao")
    if prior and Version(prior) < Version("0.16"):
        # PEFT의 LoRA 초기화가 선택 설치된 구버전 TorchAO를 검사합니다.
        # 이미 설치돼 충돌하는 런타임에서만 해당 PyTorch 버전과 맞춥니다.
        torch_ver = Version(installed("torch"))
        desired["torchao"] = "0.17.0" if torch_ver >= Version("2.11") else "0.16.0"
        print(f"이 런타임의 PEFT 호환 준비 · torch {torch_ver} · torchao {prior}", flush=True)
    needed = [f"{name}=={wanted}" for name, wanted in desired.items() if installed(name) != wanted]
    if needed:
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", *needed])


def source_details(function):
    source = escape(getsource(function))
    name = escape(function.__qualname__)
    display(HTML(f"<details><summary>{name} 실제 소스</summary><pre>{source}</pre></details>"))


def show_code(title, value):
    # 원문과 프롬프트를 줄바꿈 그대로 읽을 수 있게 표시합니다.
    display(HTML(f"<h4>{escape(title)}</h4><pre style='white-space:pre-wrap'>{escape(str(value))}</pre>"))

print(f"W13A 준비 완료 · 공개 자료 {len(bank)}건")


## 00 앱 관찰 · 생성 원문과 운영 자료

### 이론 복습
W03B의 Qwen2는 주어진 앞 토큰에서 다음 토큰 확률을 구합니다. 검증기는 생성 문자열에서 항목을 읽고 운영 자료의 값 및 ID와 비교합니다. **모델의 토큰 선택**과 **운영 자료의 사실 확인**을 화면에서 찾으세요. [W03B 8절](https://github.com/lunalab-ai/genAI/blob/2026-fall-w03b/course/notion/w03b/w03b-transformer-walkthrough.md#L174) · [W03B Colab](https://colab.research.google.com/github/lunalab-ai/genAI/blob/2026-fall-w03b/notebooks/student/w03b_transformer_walkthrough.ipynb)

$p(v \mid x_{<t})=\operatorname{softmax}(z_t)_v$에서 이번 greedy는 가장 높은 점수의 토큰을 한 개씩 붙입니다. 먼저 축소 호출을 읽으세요. 'film'은 독립영화 상영회, A는 근거 없이 생성한 Qwen2 기본 조건입니다. **TODO 00**에 화면을 보기 전 신청 표시를 예상해 기록합니다.

<details><summary>힌트 ① 값</summary>FAQ의 신청 정보가 확정됐는지 보세요.</details>
<details><summary>힌트 ② 자리</summary>toy_example은 값 하나를 받고 return 뒤 문자열을 반환합니다.</details>
<details><summary>힌트 ③ 틀</summary>prediction = "___"처럼 적습니다.</details>


In [ ]:
def toy_example(value):
    return "안내: " + value  # 전달된 값을 문장의 일부로 사용합니다.

print(toy_example("무료"))  # 입력 → 호출 → 반환 "안내: 무료"
prediction = ""  # TODO 00 · 영화의 신청란 예상
film = bank["film"]  # 독립영화 운영 메모와 FAQ를 ID로 꺼냅니다.
run0 = studio.run(film, "A")  # 실제 Qwen2의 사전 측정 재생
show_code("모델 원문", run0["raw"])
show_code("운영팀 화면의 확인된 사실", preview_view(run0, film)["facts_text"])
cards = CourseApp(studio).film_channels()
display(HTML(table_html(cards, [("label", "채널"), ("intro", "표현"), ("facts", "공통 운영 사실")])))
for card in cards:
    show_code(card["label"] + " · 모델 초안 실측 재생", card["raw"])


'안내: 무료', 원문, 운영팀의 확인된 사실이 정상 출력입니다. 키 오류가 뜨면 00 준비를 다시 실행하세요. 원문이 장황하거나 비어 있어도 그대로 둡니다.


## 01 어떤 데이터로 무엇을 학습하나

### 이론 복습
W09B의 지시 학습은 입력과 목표 응답을 짝짓습니다. W10A SFT의 **프롬프트 라벨에는 -100**, **응답 라벨에는 정답 ID**를 넣습니다. 디코더가 내부에서 다음 위치로 정렬하므로, 프롬프트 마지막 위치가 응답 첫 토큰을 예측합니다. attention_mask는 읽을 위치, labels는 손실에 넣을 정답 위치입니다. 프롬프트 자체는 읽을 수 있습니다.

[W09B 2–3절](https://github.com/lunalab-ai/genAI/blob/2026-fall-w09b/course/notion/w09b/w09b-instructions.md#L17) · [W09B 생성 Colab](https://colab.research.google.com/github/lunalab-ai/genAI/blob/2026-fall-w09b/notebooks/student/w09b_generation.ipynb) · [W10A 4절 손실 마스크](https://github.com/lunalab-ai/genAI/blob/2026-fall-w10a/course/notion/w10a/w10a-lesson.md#L203)

가상 행사 학습 24건은 응답을 학습시키고, 검증 6건은 설정을 고르는 데 씁니다. 공통 행사 2건은 통제된 결과를 나란히 읽는 데 씁니다. 서로 섞으면 운영 자료를 기억해서 맞힌 효과가 들어옵니다. 첫 행은 **사진 워크숍**입니다. prefix는 모델이 읽는 토큰, response는 출처가 붙은 목표 토큰입니다. 축소 입력은 [11,12]와 [21,22]입니다.

**TODO 01** · response_labels의 return 한 줄을 채우세요. 실행 전 미정인 신청의 목표 줄을 말해 봅니다. 정상 축소 출력은 [-100, -100, 21, 22]입니다.

<details><summary>힌트 ① 값</summary>입력 위치마다 -100, 응답 위치마다 실제 ID를 넣습니다.</details>
<details><summary>힌트 ② 자리</summary>len(prefix)개 만든 리스트를 응답 배열 앞에 둡니다.</details>
<details><summary>힌트 ③ 틀</summary>return [___] * len(___) + list(___)</details>


In [ ]:
def response_labels(prefix, response):
    return None  # TODO 01 · 프롬프트 위치와 응답 라벨을 연결하세요.

photo = bank["photo-workshop"]
example = ref["token_example"]  # 실제 Qwen2 토크나이저로 저장한 ID
show_code("사진 워크숍 학습 입력", prompt(photo, training_documents(photo)))
show_code("목표 응답", target(photo))
print("축소 라벨", response_labels([11, 12], [21, 22]))
print("실제 입력·응답 토큰 수", len(example["prefix"]), len(example["response"]))
print("분할", {split: sum(e["split"] == split for e in bank.values())
               for split in ("train", "validation", "official")})
try:
    mask_evidence = mask_witness(example, response_labels)
    print("마스크 확인 완료", mask_evidence["label_sha256"][:12])
except ValueError as exc:
    mask_evidence = None
    print("TODO 01을 확인하세요.", exc)


'신청=확인 필요|없음'과 '마스크 확인 완료'를 봅니다. 마스크 오류가 나면 프롬프트 수, 응답 순서와 리스트 길이를 축소 예로 확인하세요.


## 02 문서 검색 · 코사인과 top-k

### 이론 복습
W10A 최소 RAG에서 본 검색→문맥 구성→생성입니다. 이번에는 고정 다국어 E5를 사용합니다. query: 뒤에 질문, passage: 뒤에 운영 메모·FAQ·참고 자료를 둡니다. 유효 토큰의 은닉 상태만 평균한 뒤 길이 1로 정규화합니다. 코사인 $\cos(q,d)=q^\top d$를 384차원 내적으로 계산할 수 있습니다. 높은 순위와 특정 필드의 근거 포함 여부를 각각 확인하세요.

[W10A 6절 최소 RAG](https://github.com/lunalab-ai/genAI/blob/2026-fall-w10a/course/notion/w10a/w10a-lesson.md#L248) · [W10A 최소 RAG Colab · 문맥 비교 실습](https://colab.research.google.com/github/lunalab-ai/genAI/blob/2026-fall-w10a/notebooks/student/w10a_rag.ipynb) · [E5 모델 카드](https://huggingface.co/intfloat/multilingual-e5-small)

축소 입력 q=(1,0), 메모=(0.6,0.8), FAQ=(0.8,0.6)의 내적은 0.6과 0.8입니다. scores는 id와 score를 가진 행의 리스트이고, k는 넣을 문서 수입니다. **TODO 02** · 점수 내림차순, 동점 ID 오름차순으로 처음 k개의 ID를 반환하세요. 이 함수의 반환을 앱의 프롬프트에 연결합니다.

<details><summary>힌트 ① 값</summary>사용할 키는 score와 id입니다.</details>
<details><summary>힌트 ② 자리</summary>정렬 키로 (음수 점수, ID)를 주고 앞 k개를 자릅니다.</details>
<details><summary>힌트 ③ 틀</summary>return [r["id"] for r in sorted(scores, key=lambda r: (___, ___))[:k]]</details>


In [ ]:
def select_documents(scores, k):
    return []  # TODO 02 · 점수 순으로 문서 ID를 반환하세요.

trace = ref["vector_example"]  # 실제 E5로 계산한 북토크 벡터
z = trace["query"]             # 실제 384차원 질문 벡터
scores = [{"id": doc, "score": round(sum(qi * di for qi, di in zip(z, d)), 6)}
          for doc, d in trace["documents"].items()]  # 각 문서 벡터와 직접 내적
toy_scores = [{"id": "memo", "score": 0.6}, {"id": "faq", "score": 0.8}]
print("축소 결과. k=1", select_documents(toy_scores, 1))
print("실제 점수", scores, "차원", len(z))
print("k=1", select_documents(scores, 1), "k=2", select_documents(scores, 2))
search_ready = select_documents(scores, 2) == [r["id"] for r in trace["scores"][:2]]
if search_ready:
    reads = studio.make_inputs(bank[UPDATE_EVENT], 2, select_documents)
    show_code("모델 입력에 들어가는 문서", reads["context"])
    source_details(studio.make_inputs)
else:
    print("TODO 02를 채우면 문서 본문과 ID를 확인할 수 있습니다.")


축소 답은 'faq'입니다. 북토크의 k=1에서는 FAQ가 선택됩니다. k=2에서 추가된 운영 메모를 봅니다. 문서 순서 오류는 정렬 방향과 k를 다시 봅니다. E5를 직접 실행하려면 03의 실제 경로를 선택합니다.


## 03 기본 Qwen2는 고정하고 LoRA를 실제로 훈련

### 이론 복습
W10A의 $\Delta W=(\alpha/r)BA$를 Qwen2의 q_proj와 v_proj에 붙입니다. 입력 x에 원래 $Wx$와 저차원 경로 $(\alpha/r)BAx$를 더합니다. $r=4,\alpha=8$이면 배율은 2입니다. 과제에서 $r=2,\alpha=4$도 배율 2를 유지합니다. 270,336개와 135,168개는 실제 모형의 학습 가능 스칼라 수입니다. 기본 가중치는 동결하고, AdamW 0.0005, batch 1, seed 42, dropout 0, FP32로 응답 손실을 최소화합니다. [W10A 2절 LoRA와 E1–E3](https://github.com/lunalab-ai/genAI/blob/2026-fall-w10a/course/notion/w10a/w10a-lesson.md#L30) · [W10A LoRA Colab](https://colab.research.google.com/github/lunalab-ai/genAI/blob/2026-fall-w10a/notebooks/student/w10a_adaptation.ipynb) · [PEFT 설정](https://huggingface.co/docs/peft/main/en/package_reference/lora)

축소 예로 정답 토큰 확률이 0.25에서 0.5로 증가하면 그 위치의 교차 엔트로피 $-\ln p$는 약 1.39에서 0.69가 됩니다. **TODO 03** · 실제 실습이면 RUN_CLASS_LIVE를 True로 바꾸고 실행하세요. 아래 실측처럼 동일한 한 입력에서 첫 토큰의 확률과 손실을 비교합니다. 학습 4회 전체 손실은 서로 다른 행사에서 측정합니다. 이 값만으로 검증 성능을 결론내릴 수 없습니다.

<details><summary>힌트 ① 값</summary>수업에서 4회, 과제에서 훈련 행사 24개를 한 번씩 사용합니다.</details>
<details><summary>힌트 ② 자리</summary>fit의 rows는 train, mask는 01에서 작성한 response_labels입니다.</details>
<details><summary>힌트 ③ 틀</summary>fit(train, 저장폴더, rank=4, steps=4, mask=response_labels)</details>


In [ ]:
RUN_CLASS_LIVE = False  # TODO 03 · 직접 CPU 실행은 True
class_mode = "reference"
class_fit = ref["training"]["4:4"]  # 같은 조건을 실제로 실행해 저장한 결과
first = ref["first_token"]
if RUN_CLASS_LIVE and mask_evidence and search_ready:
    # 실제 경로의 의존성을 명시합니다. GPU, 유료 키, 임시 공개 주소가 필요 없습니다.
    prepare_live_packages()
    print("Qwen2와 E5를 준비합니다. 새 런타임은 모델 다운로드부터 시작합니다.")
    model = studio.runtime
    saved = directory / "my-class-adapter"
    class_fit = model.fit(train, saved, rank=4, steps=4, mask=response_labels)
    class_mode = "live"
    first = {"base": model.first_token(example), "after": model.first_token(example, 4, 4)}
    show_code("지금 계산한 E5 순위", studio.ranking(bank[UPDATE_EVENT], live=True))
elif RUN_CLASS_LIVE:
    print("실제 학습을 하려면 TODO 01·02를 완성하세요. 현재 사전 실측 표시입니다.")

from luna_genai.design_models import QwenDesign
source_details(QwenDesign.fit)  # fit에 들어가는 마스크 → loss → backward → step → 저장
print("실행 경로", class_mode, "기본 해시 동일", class_fit["base_before"] == class_fit["base_after"])
print("학습 가능 스칼라", class_fit["trainable"], "바뀐 텐서", class_fit["changed_tensors"])
show_code("4회 업데이트와 응답 손실", "\n".join(
    f"step {x['step']} | {x['event']} | loss {x['loss']}" for x in class_fit["updates"]
))
show_code("같은 목표 토큰의 확률과 손실", json.dumps(first, ensure_ascii=False, indent=2))


**정상 출력** · 기본 해시 동일 True, 바뀐 텐서 96, loss 네 줄, 동일 목표 토큰의 확률 두 개입니다. 04에서 C를 실행할 때 어댑터 파일을 다시 읽습니다. 실제 다운로드·학습 오류 시 오류 화면을 남기고 RUN_CLASS_LIVE를 False로 바꿔 이 셀을 다시 실행하세요. 경로 라벨이 reference로 바뀝니다.


## 04 어떤 변수가 어떤 오류를 바꿨나

### 이론 복습
W10A E8의 통제 비교와 평가를 적용합니다. 북토크는 훈련하지 않은 행사입니다. 같은 Qwen2와 생성 형식, greedy 최대 180을 유지합니다. **A→B는 근거만, B→C는 저장한 어댑터만** 바뀝니다. 동일한 프롬프트 SHA는 B/C에 같은 문자열이 들어갔음을 확인하는 표시입니다.

[W10A 5절 평가](https://github.com/lunalab-ai/genAI/blob/2026-fall-w10a/course/notion/w10a/w10a-lesson.md#L220) · [W10A E8 문맥 통제](https://github.com/lunalab-ai/genAI/blob/2026-fall-w10a/course/notion/w10a/w10a-lesson.md#L275)

일시·장소·참가비 중 두 개가 맞으면 2/3입니다. FAQ의 신청이 비어 있을 때 '확인 필요'면 미정 처리, 값을 만들면 자료 밖 값, 아무 줄도 없으면 출력 누락입니다. 맞은 값에 엉뚱한 ID를 붙이면 값은 맞아도 출처 검사를 통과하지 못합니다. **TODO 04** · 비교할 항목 field_to_read에 when/where/fee/signup/capacity 중 한 가지를 적습니다.

<details><summary>힌트 ① 값</summary>신청을 확인하려면 signup을 씁니다.</details>
<details><summary>힌트 ② 자리</summary>field_rows가 돌려준 리스트에서 field 키를 찾습니다.</details>
<details><summary>힌트 ③ 틀</summary>field_to_read = "___"</details>


In [ ]:
field_to_read = "signup"  # TODO 04 · 별도로 확인할 필드
case = bank[UPDATE_EVENT]  # 북토크는 검증 분할에 있습니다.
if search_ready:
    # 02에서 만든 검색 함수가 여기와 05·06 파이프라인에 들어갑니다.
    section4 = [studio.run(case, c, mode=class_mode, selection=select_documents)
                for c in ("A", "B", "C")]
    print("경로", class_mode, "B/C 동일 프롬프트", section4[1]["prompt_sha256"] == section4[2]["prompt_sha256"])
    print("C 어댑터 다시 읽음", section4[2]["adapter_reloaded"], section4[2]["adapter_sha256"][:12])
    for run in section4:
        print(LABELS[run["condition"]], run["used_docs"], run["input_tokens"])
        print([row for row in field_rows(run, case) if row["field"] == field_to_read])
    show_code("C 생성 원문", section4[2]["raw"])
    show_code("C에 실제 입력한 프롬프트", section4[2]["prompt"])
    source_details(field_rows)
else:
    section4 = []
    print("02 문서 선택을 완성한 뒤 04를 재실행하세요.")


**정상 출력** · A의 문서 ID 없음, B/C의 문서 ID 동일, C 어댑터 다시 읽음 True. 원문이 틀린 경우 선택한 필드·원문 값·근거 문서가 입력에 있었는지 구분합니다. 생성에 실패하면 03의 실행 경로, 어댑터 파일과 02의 선택을 확인합니다.


## 05 개인 D 실험과 기본값 선택

### 이론 복습
한 번에 한 변수를 바꿔 변화의 원인을 좁힙니다. **top_k**를 고르면 C의 r=4를 유지하고 문서 수만 2→1로 줄입니다. **lora_rank**는 문서 2개와 $\alpha/r=2$를 유지하고 rank만 4→2로 줄입니다. 검증 행사 6건에서 B/C/D를 비교합니다. 두 공식 사례 A/B/C는 별도로 보존합니다. [W10A 2절 LoRA 설정](https://github.com/lunalab-ai/genAI/blob/2026-fall-w10a/course/notion/w10a/w10a-lesson.md#L99) · [W10A E8 통제 비교](https://github.com/lunalab-ai/genAI/blob/2026-fall-w10a/course/notion/w10a/w10a-lesson.md#L275)

**TODO 05a** · axis를 고르고 가설을 한 문장으로 적습니다. 예를 들어 'k를 줄이면 FAQ에 들어 있는 신청 근거가 줄어들 것이다.' 실제 표로 확인합니다. **TODO 05b** · 선택 함수의 정렬 키 한 줄을 완성합니다. 최솟값 기준으로 (자료 밖 값 수, 음수 핵심 정답 수, 음수 유효 출처 수, 입력 토큰 수, 학습 가능 스칼라, 조건 문자)입니다. 끝까지 같으면 B→C→D 순입니다.

| 키 | 실제 단위 |
|---|---|
| unsupported, core_correct, valid_sources | 미정 자료 밖 값, 핵심 값 일치, 값과 출처가 함께 맞은 필드 수 |
| input_tokens, adapter_params | 검증 6건의 입력 토큰 총합과 어댑터의 학습 가능 스칼라 개수 |

<details><summary>힌트 ① 값</summary>숫자가 클수록 좋은 두 항목에만 음수를 붙입니다.</details>
<details><summary>힌트 ② 자리</summary>decision_key의 return 안에 정해진 순서로 여섯 키를 넣습니다.</details>
<details><summary>힌트 ③ 틀</summary>return (row["unsupported"], -row[___], ..., row["condition"])</details>


In [ ]:
axis = "top_k"  # TODO 05a · top_k 또는 lora_rank
hypothesis = ""  # TODO 05a · 표에서 직접 확인할 예측 한 문장
HOMEWORK_LIVE = False  # 과제 실제 24회 훈련/생성을 원하면 True. 느리면 False로 다시 실행하세요.

def decision_key(row):
    return ()  # TODO 05b · 수업의 여섯 기준을 정렬 키로 만드세요.

def pick(rows):
    return min(rows, key=decision_key)["condition"]  # 학생의 정렬 기준으로 실제 앱 기본값 선택

def run_design(axis, hypothesis, *, live=False):
    app = CourseApp(studio)
    if live:
        from luna_genai.design_models import QwenDesign
        # 03을 건너뛰었어도 실제 과제 실행의 의존성 설치 위치를 동일하게 둡니다.
        prepare_live_packages()
        if studio._runtime is None:
            studio._runtime = QwenDesign()
        for rank in ((4, 2) if axis == "lora_rank" else (4,)):
            print("훈련 중", rank, "rank · 24회", flush=True)
            studio.runtime.fit(train, directory / f"homework-r{rank}", rank=rank, steps=24, mask=response_labels)
        print("두 공통 행사와 검증 6건의 실제 생성 중", flush=True)
    stats = app.evaluate(axis, hypothesis, mode="live" if live else "reference",
                         selection=select_documents, chooser=pick, mask=response_labels)
    numeric_columns = [
        ("label", "조건"), ("retrieved", "찾은 근거"), ("retrieval_total", "전체 근거"),
        ("core_correct", "핵심 정답"), ("core_total", "핵심 분모"), ("unsupported", "자료 밖 값"),
        ("missing", "출력 누락"), ("valid_sources", "유효 출처"),
        ("input_tokens", "입력 토큰 합"), ("adapter_params", "어댑터 스칼라"),
    ]
    display(HTML("<h4>검증 6건 / 설정</h4>" + table_html(stats["rows"], numeric_columns)))
    print("앱 기본값", stats["choice"], "경로", app.manifest["mode"])
    return app

project = None
design_ready = (mask_evidence and search_ready and hypothesis.strip()
                and len(decision_key({"unsupported": 0, "core_correct": 0, "valid_sources": 0,
                                      "input_tokens": 0, "adapter_params": 0, "condition": "B"})) == 6)
if design_ready:
    project = run_design(axis, hypothesis, live=HOMEWORK_LIVE)
else:
    print("01·02와 TODO 05a·05b를 완료하세요. 실험과 내보내기는 완료 뒤 열립니다.")


정상 출력은 B/C/D 세 행, 각 핵심 기준 분모 18, 앱 기본값 한 개입니다. 키 오류가 나면 표의 키 이름부터 대조합니다. 실제 과제 학습은 순서대로 24회씩 실행합니다. 참고 경로에서는 같은 CPU 설정에서 측정해 둔 기록으로 비교하고 점수 기준은 같습니다.


## 06 학생의 선택을 실제 앱 기본값으로 연결

### 이론 복습
데이터와 모델의 버전은 서로 다른 값입니다. 문서를 갱신해도 기본 Qwen2와 어댑터가 즉시 바뀌지는 않습니다. 검증에서 선택한 설정을 유지하고 운영 메모만 갱신하면 **설정 ID는 같고 문서 버전은 달라집니다**. 갱신 문서가 검색 결과에 들어와야 프롬프트 안의 값도 바뀝니다. W12A가 강조한 출력 유형에 맞는 검증과도 연결됩니다. [W12A 10.10 평가](https://github.com/lunalab-ai/genAI/blob/2026-fall-w12a/course/notion/w12a/w12a-lesson.md#L98)

**TODO 06** · before_predictions에 예상 참가비와 '문서 버전만/프롬프트도' 중 예상 결과를 적습니다. **앱 기본값 적용 → 참가비 2,000원 공지 반영** 두 버튼을 순서대로 눌러 실제 표시와 대조합니다. 표는 운영 자료 값과 모델 생성값을 구분합니다.

<details><summary>힌트 ① 값</summary>원본 북토크의 참가비는 무료입니다. 갱신값은 2,000원입니다.</details>
<details><summary>힌트 ② 자리</summary>앱의 두 ID와 실제 사용 문서 목록을 확인합니다.</details>
<details><summary>힌트 ③ 틀</summary>before_predictions = ("___ → ___", "___")</details>


In [ ]:
before_predictions = ("", "")  # TODO 06 · 참가비·프롬프트 변화 예상
if project:
    buttons = show_widget(project)  # 앞에서 완성한 학생 함수와 선택을 사용합니다.
    # 모드·설정·문서·프롬프트가 실제 화면의 상태 값입니다.
    initial_state, revised_state = project.manifest["app_state"]
    print("설정 유지", initial_state["config_id"] == revised_state["config_id"])
    print("문서 갱신", initial_state["document_version"] != revised_state["document_version"])
    print("프롬프트 변경", initial_state["prompt_sha256"] != revised_state["prompt_sha256"])
    print("입력 문서", initial_state["used_docs"], "→", revised_state["used_docs"])
else:
    print("05의 개인 실험이 끝나면 여기에 앱이 열립니다.")


정상 화면의 운영 자료에는 무료에서 2,000원으로 바뀝니다. k가 작아 갱신한 메모를 검색하지 못했다면 프롬프트는 그대로일 수 있습니다. 모델 원문을 수동 수정하지 말고 그 차이를 판정에서 찾으세요. 버튼이 회색이면 바로 앞 단계와 위젯 출력이 완료됐는지 확인합니다.


## 07 검증 보고 · 두 오류를 짚고 ZIP 제출

### 이론 복습
W10A 5절은 PPL과 출력 평가의 일을 구분합니다. 이 과제는 운영 자료로 **항목별로 재현 가능한 판정**을 합니다. 오류를 찾으면 원문이 정말 빠졌는지, 놓친 문서가 무엇인지 따로 봅니다. 화면에서 선택한 설정과 파일의 설정 ID는 동일해야 합니다. [W10A 5절](https://github.com/lunalab-ai/genAI/blob/2026-fall-w10a/course/notion/w10a/w10a-lesson.md#L220)

표의 번호를 하나 골라 코드에서 자세히 열 수 있습니다. 두 건은 서로 다른 실행에서 고릅니다. **TODO 07** · 번호, 필드, 판정, 출처 유효(bool), 입력에 해당 근거가 있었는지(bool)를 적습니다. 수업 중에는 예시 하나를 읽고 과제에서 완성해도 됩니다.

<details><summary>힌트 ① 값</summary>필드는 when/where/fee/signup/capacity, 판정은 mismatch/missing/unsupported/unknown_ok/match 중에서 고릅니다.</details>
<details><summary>힌트 ② 자리</summary>review_case에 번호를 지정해 선택한 원문과 넣은 문서를 펼칩니다. field_view에서 값을 대조합니다.</details>
<details><summary>힌트 ③ 틀</summary>notes = [(번호, "필드", "판정", False, True), (번호, ...)]</details>


In [ ]:
def submitted_cases(app):
    # 공식 6건 + 검증 18건에 학생이 읽기 쉬운 1부터의 번호를 붙입니다.
    return app.manifest["official"] + app.manifest["validation"]

def review_case(app, number):
    record = submitted_cases(app)[number - 1]  # 사용자 표시 번호를 0부터의 인덱스로 변환
    event = bank[record["event"]]
    values = preview_view(record, event)
    print("선택", number, event["name"], LABELS[record["condition"]], "입력 ID", record["used_docs"])
    show_code("생성 원문", record["raw"])
    show_code("원문 문서", format_sources(event["docs"]))
    display(HTML(table_html(values["field_view"], [
        ("field", "항목"), ("generated", "생성값"), ("expected", "운영 자료"), ("status", "판정"),
        ("source_ok", "출처 유효"), ("evidence_seen", "근거가 입력에 있었는가"),
    ])))

if project:
    for number, row in enumerate(submitted_cases(project), 1):
        print(number, bank[row["event"]]["name"], LABELS[row["condition"]])
    review_case(project, 1)  # 번호를 바꾸어 원문을 펼치세요.
notes = []  # TODO 07 · 서로 다른 번호의 오류 두 건을 기록하세요.


**정상 출력** · 24개 실행 번호, 선택한 행사의 원문과 문서, 다섯 필드 판정입니다. IndexError가 나면 1–24 사이 번호를 사용하세요. **아래 ZIP 셀 전에** 06의 두 버튼을 모두 누릅니다. 캡처가 어렵더라도 ZIP은 저장되며 채점 기준은 같습니다.


In [ ]:
assert len(bank) == 32 and len(train) == 24 and len(trace["query"]) == 384
assert all(abs(x["score"] - next(r["score"] for r in trace["scores"] if r["id"] == x["id"])) < 1e-5
           for x in scores)
assert class_fit["base_before"] == class_fit["base_after"] and class_fit["changed_tensors"] > 0
assert class_fit["trainable"] == 270336 and len(class_fit["updates"]) == 4

def package(app, notes):
    from luna_genai.event_design import choose_default

    records = submitted_cases(app)
    if len(notes) != 2 or notes[0][0] == notes[1][0]:
        raise ValueError("서로 다른 실행에서 오류 두 개를 고르세요.")
    if choose_default(summarize(app.manifest["validation"], bank)) != app.manifest["choice"]:
        raise ValueError("05b의 정렬 기준을 검증표와 다시 대조하세요.")
    observations = []
    for n, field, status, source_ok, evidence_seen in notes:
        if not 1 <= n <= len(records):
            raise ValueError("실행 번호는 1부터 24까지입니다.")
        raw = records[n - 1]
        row = next(item for item in field_rows(raw, bank[raw["event"]]) if item["field"] == field)
        if (status, source_ok, evidence_seen) != (row["status"], row["source_ok"], row["evidence_seen"]):
            raise ValueError(f"{n}번의 원문, 출처와 실제 입력 문서를 다시 확인하세요.")
        if row["status"] in ("match", "unknown_ok") and row["source_ok"]:
            raise ValueError(f"{n}번은 정상 사례입니다. 오류 또는 출처 문제를 고르세요.")
        observations.append({"run_id": raw["run_id"], "field": field, "status": status,
                             "source_ok": source_ok, "evidence_seen": evidence_seen})
    zipped = app.zip_bytes(observations)
    with ZipFile(BytesIO(zipped)) as upload:
        manifest = json.loads(upload.read("run.json"))
        assert len(json.loads(upload.read("generations.json"))) == 26
        assert len(upload.read("validation.csv").decode("utf-8").splitlines()) == 131
        assert len(upload.read("summary.csv").decode("utf-8").splitlines()) == 7
        print("자기검사 완료 · 생성 26건 · 검증 130행 · 요약 6행 · 화면", manifest["capture"])
    return zipped

if project and len(notes) == 2 and project.updated:
    zip_path = directory / "w13a-design-submission.zip"
    zip_path.write_bytes(package(project, notes))
    print("제출 ZIP", zip_path)
    import ipywidgets as widgets
    download = widgets.Button(description="과제 ZIP 다운로드", button_style="success")

    def download_zip(_):
        try:
            from google.colab import files
        except ImportError:
            print(zip_path)
        else:
            files.download(str(zip_path))

    download.on_click(download_zip)
    display(download)
else:
    print("06 앱 적용·갱신, 07 두 오류를 확인한 뒤 이 셀을 다시 실행합니다.")


## 다음 차시로 이어가기

공지에서 확인한 행사 ID, 일시·장소·참가비·미정 상태를 팀의 데이터로 보관합니다. 다음 차시에는 이 값을 포스터 레이아웃과 음성 안내 스크립트에 씁니다. 후속 결과도 같은 행사 사실과 대조합니다. [W11A 이미지 편집](https://github.com/lunalab-ai/genAI/releases/tag/2026-fall-w11a) · [W11B 음성](https://github.com/lunalab-ai/genAI/releases/tag/2026-fall-w11b-r1)
